# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring a Croissant-formatted dataset using the `mlcroissant` library. We'll access and process the FAIR^2 dataset documenting mass spectrometry analysis of extracellular vesicles from human mast cells.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Citation: {getattr(metadata, 'citeAs', '')}\n")
print(f"License: {getattr(metadata, 'license', '')}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** In Croissant, a 'record set' is a logical table or set of tabular records. Each record set has an `@id` that you need to use to reference and extract its data with `mlcroissant`.

In [ ]:
# List available record sets and their fields using their @id
print("Available record sets (by @id):")
for record_set in getattr(metadata, 'record_sets', []):
    print(f"- Record set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '')}")
    print(f"  Description: {record_set.get('description', '')}")
    print(f"  Fields: ")
    for field in record_set.get('fields', []):
        print(f"    - Field @id: {field['@id']}")
        print(f"      Name: {field.get('name','')}")
        print(f"      Data type: {field.get('dataType','')}")
    print()

# Show all @id for reference
record_set_ids = [rs['@id'] for rs in getattr(metadata, 'record_sets', [])]
print('Record Set @ids:', record_set_ids)
# If there are no record_sets at the top-level, try introspecting fields using dataset API
if not record_set_ids:
    try:
        print("No record_sets property in metadata; attempting to explore actual record sets via dataset API...")
        all_record_sets = dataset.record_sets
        for rs in all_record_sets:
            print(f"Found record set: {rs['@id']} (name: {rs.get('name','')})")
        record_set_ids = [rs['@id'] for rs in all_record_sets]
    except Exception as e:
        print("Could not infer record set ids automatically.", e)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we load all record sets into individual Pandas DataFrames (one per record set).

In [ ]:
# Extract all record set IDs from the dataset
try:
    # Use dataset.record_sets for mlcroissant >= 1.1
    record_sets_info = dataset.record_sets
    record_set_ids = [rs['@id'] for rs in record_sets_info]
except AttributeError:
    # Fallback if attribute differs
    record_set_ids = []

dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for this record set
    try:
        records_iterator = dataset.records(record_set=record_set_id)
        records = list(records_iterator)
        if len(records) == 0:
            print(f"No data found for record set {record_set_id}")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

if dataframes:
    # Display the first DataFrame loaded
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"First record set ID: {selected_record_set_id}")
    print(f"Columns: {dataframes[selected_record_set_id].columns.tolist()}")
    display(dataframes[selected_record_set_id].head())
else:
    print("No DataFrames could be built from available record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes typical EDA tasks such as removing outliers, transforming data, and grouping records.

*Note:* In this example, we'll demonstrate EDA on the first loaded table.

In [ ]:
import numpy as np

if dataframes:
    df = dataframes[selected_record_set_id]
    print(f"Columns in selected table ({selected_record_set_id}): {df.columns.tolist()}")

    # Attempt to auto-detect a numeric column (e.g. those of type float or int)
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    if len(numeric_columns) == 0:
        # Try to coerce some likely columns to numeric using a heuristic
        candidate_cols = [col for col in df.columns if any(token in col.lower() for token in ["mw", "weight", "abundance", "count", "coverage", "score", "number"])]
        for col in candidate_cols:
            # Try to convert to numeric
            df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_columns = df.select_dtypes(include=[np.number]).columns

    if len(numeric_columns) > 0:
        numeric_field_id = numeric_columns[0]
        print(f"Selected numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} above mean ({threshold:.2f}): {len(filtered_df)} rows out of {len(df)}")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely categorical field (e.g. any with a small unique count, or name/label)
        candidate_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < 10 and df[col].dtype == object]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(grouped_df)
        else:
            print("No suitable group field for grouping found.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No record set DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the main numeric field in the first available record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if dataframes and len(numeric_columns) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to plot a numeric distribution.")

## 6. Conclusion

In this notebook, we demonstrated how to access and explore a FAIR-compliant dataset with the `mlcroissant` library. We:
- Loaded dataset metadata and identified available record sets and field `@id`s
- Loaded records for each record set into DataFrames for analysis
- Performed basic exploratory data analysis (EDA), including filtering, normalization, and grouping
- Visualized the distribution of key numeric variables

For more advanced analysis, customize the EDA section or reference fields and record sets by their `@id` as needed using the documentation from the Croissant schema.